In [6]:
# ============================================================
# REPOSITORY SETUP — YOUR GitHub repository
# ============================================================

from pathlib import Path
import shutil
import urllib.request
import zipfile

ZIP_URL = "https://codeload.github.com/Imvixh/flyrank-ml-internship/zip/refs/heads/main"

ROOT = Path("/content/flyrank-ml-internship")
ZIP_PATH = Path("/content/flyrank-ml-internship-main.zip")

# Remove stale/incomplete runtime copy
if ROOT.exists():
    shutil.rmtree(ROOT)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print("Downloading YOUR GitHub repository...")
urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)

print("Extracting repository...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall("/content")

EXTRACTED = Path("/content/flyrank-ml-internship-main")
EXTRACTED.rename(ROOT)

ZIP_PATH.unlink(missing_ok=True)

DATA = ROOT / "data/raw/content_refresh_anonymized.csv"

print("\n" + "=" * 55)
print("REPOSITORY CHECK")
print("=" * 55)
print("Repository exists:", ROOT.exists())
print("Dataset exists:", DATA.exists())
print("Repository:", ROOT)
print("Dataset:", DATA)

if not ROOT.exists():
    raise FileNotFoundError("Repository could not be prepared.")

if not DATA.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA}")

print("\n✓ YOUR repository ready")
print("✓ Dataset ready")

Extracting repository...

REPOSITORY CHECK
Repository exists: True
Dataset exists: True
Repository: /content/flyrank-ml-internship
Dataset: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

✓ YOUR repository ready
✓ Dataset ready


# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature vector

The feature vector uses observable content and search-performance signals that are available for a page before the prediction/review decision. Numeric variables are converted to numeric values and missing numeric values are handled with median imputation. Categorical variables are one-hot encoded with unknown categories ignored.

Identifiers such as `content_id` and `client_id` are not used as predictive features. Outcome-related fields such as `trend_direction` and `trend_pct` are excluded to reduce leakage risk.

In [7]:
# ============================================================
# SECTION 1 — BUILD THE FEATURE VECTOR
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Load the dataset
df = pd.read_csv(DATA)

# Candidate predictive features
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
]

# Outcome / identifier fields deliberately excluded
excluded_features = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
]

# Verify all required fields exist
required = numeric_features + categorical_features + excluded_features
missing_columns = [c for c in required if c not in df.columns]

print("Missing required columns:", missing_columns)

assert not missing_columns, f"Missing columns: {missing_columns}"

# Create X
X = df[numeric_features + categorical_features].copy()

# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combined preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

# Build feature matrix
X_vector = preprocessor.fit_transform(X)

feature_names = preprocessor.get_feature_names_out()

X_vector = pd.DataFrame(
    X_vector,
    columns=feature_names,
    index=df.index
)

print("\nOriginal rows:", len(df))
print("Feature matrix shape:", X_vector.shape)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nMissing values after preprocessing:",
      int(X_vector.isna().sum().sum()))

display(X_vector.head())

Missing required columns: []

Original rows: 30000
Feature matrix shape: (30000, 25)
Numeric features: 15
Categorical features: 3

Missing values after preprocessing: 0


,numeric__search_volume,numeric__competition,numeric__cpc,numeric__word_count,numeric__char_count,numeric__content_age_days,numeric__days_since_last_update,numeric__impressions_90d,numeric__clicks_90d,numeric__sessions_90d,...,categorical__competition_level_HIGH,categorical__competition_level_LOW,categorical__competition_level_MEDIUM,categorical__content_type_comparison article,categorical__content_type_feedly article,categorical__content_type_keyword article,categorical__main_intent_commercial,categorical__main_intent_informational,categorical__main_intent_navigational,categorical__main_intent_transactional
0,10.0,0.67,2.05,3221.0,20457.0,187.0,20.0,3803.0,29.0,17.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,90.0,0.01,0.05,2481.0,15562.0,445.0,25.0,15320.0,7.0,9.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,0.0,0.00,0.00,3515.0,23643.0,141.0,20.0,12581.0,11.0,11.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,10.0,0.00,0.00,2877.0,19116.0,463.0,22.0,11751.0,58.0,78.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
4,0.0,0.00,0.00,2803.0,17469.0,263.0,14.0,19140.0,24.0,145.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

The numeric features represent search demand, content size, content age, historical search performance, and engagement. Missing numeric values are handled with median imputation. Categorical features describe competition, content type, and search intent; missing categorical values are filled with the most frequent category and then one-hot encoded.

The identifiers `content_id` and `client_id` are used only for traceability/grouping and are not predictive features.

The intended prediction point is before a human refresh decision is made. Therefore, outcome-defining fields such as `trend_direction` and `trend_pct` are not included in the feature vector.

The 90-day performance fields require particular caution: they are historical observed measurements and must only be used if their measurement window is available before the prediction point. They should not be interpreted as future information.

In [8]:
# ============================================================
# SECTION 2 — FEATURE NOTES / AVAILABILITY CHECK
# ============================================================

feature_notes = pd.DataFrame({
    "feature": numeric_features + categorical_features,
    "type": (
        ["numeric"] * len(numeric_features)
        + ["categorical"] * len(categorical_features)
    ),
    "missing_pct": [
        round(df[c].isna().mean() * 100, 2)
        for c in numeric_features + categorical_features
    ],
    "contains_90d_window": [
        "90d" in c.lower()
        for c in numeric_features + categorical_features
    ],
})

display(feature_notes)

print("\nFeature count:", len(feature_notes))
print("Numeric:", (feature_notes["type"] == "numeric").sum())
print("Categorical:", (feature_notes["type"] == "categorical").sum())

print("\nFeatures with 90-day windows:")
print(
    feature_notes.loc[
        feature_notes["contains_90d_window"], "feature"
    ].tolist()
)

,feature,type,missing_pct,contains_90d_window
0,search_volume,numeric,8.23,False
1,competition,numeric,8.23,False
2,cpc,numeric,8.23,False
3,word_count,numeric,25.66,False
4,char_count,numeric,25.66,False
5,content_age_days,numeric,0.00,False
6,days_since_last_update,numeric,0.00,False
7,impressions_90d,numeric,0.00,True
8,clicks_90d,numeric,0.00,True
9,sessions_90d,numeric,0.00,True



Feature count: 18
Numeric: 15
Categorical: 3

Features with 90-day windows:
['impressions_90d', 'clicks_90d', 'sessions_90d']


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

I checked the feature vector for fields that directly define the target, identify the record or client, or may represent information that is not safely available at the prediction point.

`trend_direction` and `trend_pct` are excluded because they directly describe the outcome. `content_id` and `client_id` are excluded from predictive features because identifiers can allow memorization rather than learning meaningful patterns.

The 90-day performance variables are treated as historical signals and require temporal interpretation. They should only be used when their measurement window is known to precede the prediction point. The checks below verify that outcome-defining fields are absent from the feature vector.

In [9]:
# ============================================================
# SECTION 3 — LEAKAGE HUNT
# ============================================================

print("=== LEAKAGE CHECK ===")

# 1. Outcome fields must not be in the feature vector
outcome_fields = ["trend_direction", "trend_pct"]

for col in outcome_fields:
    assert col not in X.columns, f"LEAKAGE: {col} is in X"

print("✓ Outcome fields excluded:", outcome_fields)

# 2. Identifiers must not be predictive features
identifier_fields = ["content_id", "client_id"]

for col in identifier_fields:
    assert col not in X.columns, f"IDENTIFIER LEAKAGE: {col} is in X"

print("✓ Identifier fields excluded:", identifier_fields)

# 3. Verify feature-vector columns
vector_columns = set(X.columns)

for col in excluded_features:
    assert col not in vector_columns, f"Excluded field found: {col}"

print("✓ All explicitly excluded fields absent from feature vector")

# 4. Check for suspicious target-like names
suspicious = [
    c for c in X.columns
    if any(term in c.lower() for term in ["trend", "target", "label", "outcome"])
]

print("\nSuspicious target-like feature names:")
print(suspicious)

assert not suspicious, f"Potential target leakage found: {suspicious}"

print("\n✓ Leakage name check passed")

=== LEAKAGE CHECK ===
✓ Outcome fields excluded: ['trend_direction', 'trend_pct']
✓ Identifier fields excluded: ['content_id', 'client_id']
✓ All explicitly excluded fields absent from feature vector

Suspicious target-like feature names:
[]

✓ Leakage name check passed


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

| Field | Why excluded |
|---|---|
| `content_id` | Identifier only; using it as a feature could allow memorization instead of learning general patterns. |
| `client_id` | Identifier/grouping field; excluded from predictive features to avoid client-specific memorization and support honest validation. |
| `trend_direction` | Directly describes the outcome used to define decline, so including it would cause target leakage. |
| `trend_pct` | Directly measures the trend outcome and is therefore excluded from the predictive feature vector. |

These exclusions keep the feature vector focused on observable content, demand, performance, and engagement signals rather than identifiers or outcome-defining information.

In [10]:
# ============================================================
# SECTION 4 — VERIFY EXCLUDED FIELDS
# ============================================================

print("Excluded fields and reasons:")

exclusion_reasons = {
    "content_id": "Identifier; prevents memorization.",
    "client_id": "Identifier/grouping field; prevents client-specific memorization.",
    "trend_direction": "Outcome-defining field; prevents target leakage.",
    "trend_pct": "Outcome measurement; prevents target leakage.",
}

for field, reason in exclusion_reasons.items():
    print(f"\n{field}: {reason}")

print("\n=== FINAL FEATURE VECTOR CHECK ===")

print("Number of raw predictive fields:", len(numeric_features + categorical_features))
print("Number of transformed features:", X_vector.shape[1])

for field in excluded_features:
    assert field not in X.columns
    assert field not in X_vector.columns

print("✓ All excluded fields are absent from the feature vector")
print("✓ Feature vector contains no identifiers")
print("✓ Feature vector contains no direct outcome fields")

Excluded fields and reasons:

content_id: Identifier; prevents memorization.

client_id: Identifier/grouping field; prevents client-specific memorization.

trend_direction: Outcome-defining field; prevents target leakage.

trend_pct: Outcome measurement; prevents target leakage.

=== FINAL FEATURE VECTOR CHECK ===
Number of raw predictive fields: 18
Number of transformed features: 25
✓ All excluded fields are absent from the feature vector
✓ Feature vector contains no identifiers
✓ Feature vector contains no direct outcome fields


## Self-check — VERIFIED

- [x] Every section above is filled — markdown thinking and supporting code are complete.
- [x] The notebook runs top to bottom with no errors — verified using Runtime → Run all.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful words such as observed, measured, directional, and decision-support.
- [x] Feature vector was built with numeric imputation and categorical encoding.
- [x] Leakage checks were completed and outcome/identifier fields were excluded.
- [x] The completed notebook was committed to my repository under `work/notebooks/`.